In [1]:
!pip install sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 7.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ------ --------------------------------- 1.8/10.6 MB 10.2 MB/s eta 0:00:01
   -------------- ------------------------- 3.9/10.6 MB 10.4 MB/s eta 0:00:01
   --------------------- ------------------ 5.8/10.6 MB 9.7 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.6 MB 8.5 MB/s eta 0:00:01
   ------------------------------ --------- 8.1/10.6 MB 8.1 MB/s eta 0:00:01
   ----------------------------------- ---- 9.4/10.6 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 7.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 663.6/663.6 kB 6.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import os
import faiss
import pickle

from sentence_transformers import SentenceTransformer

In [3]:
DATA_PATH = r"D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum\processed\chunked_data.csv"

chunks_df = pd.read_csv(DATA_PATH)

chunks_df.head()

,subject,file_name,chunk_id,chunk_text,chunk_length
0,data_management,0-CommonIntro.pdf,0,data management 2 teaching team 1 course goals...,698
1,data_management,0-CommonIntro.pdf,1,"d. b. meysman, and mohamed ali. introducing da...",698
2,data_management,0-CommonIntro.pdf,2,for the project the project must be approved ...,699
3,data_management,0-CommonIntro.pdf,3,you must present a new project spotywhy social...,194
4,data_management,1-DataLifeCycle.pdf,0,data lifecycle 2 methodologies 1 tools 2 phase...,699


# CELL 4 — LOAD EMBEDDING MODEL
### Why?

### lightweight
### fast
### strong semantic performance
### very common in RAG research

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\lenovo\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
sample_text = chunks_df.iloc[0]["chunk_text"]

embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [6]:
texts = chunks_df["chunk_text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

In [9]:
print(type(embeddings))

print(embeddings.shape)

<class 'numpy.ndarray'>
(4619, 384)


### STEP 4.1 — CREATE FAISS INDEX

In [10]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [11]:
print(index.ntotal)

4619


In [12]:
VECTOR_PATH = r"D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum\vector_db"

os.makedirs(VECTOR_PATH, exist_ok=True)

In [32]:
faiss.write_index(
    index,
    os.path.join(VECTOR_PATH, "faiss_index.index")
)

print("FAISS index saved!")

FAISS index saved!


In [40]:
import pickle
import os

with open(os.path.join(VECTOR_PATH, "metadata.pkl"), "rb") as f:
    chunks = pickle.load(f)

print("Loaded chunks:", len(chunks))

Loaded chunks: 4619


In [41]:
print(chunks[0]["chunk_text"][:300])

data management 2 teaching team 1 course goals and organization 2 exam rules 3 experience from the past 4 teaching team  data management  prof. andrea maurino (lead professor) andrea.maurinounimib.it schedule  see the calendar  november 25 and 26 there will be recorded lecturs  in-presence and regis


### STEP 4.2 — TEST SEMANTIC SEARCH

In [34]:
def semantic_search(query, top_k=5):

    # query embedding
    query_embedding = model.encode([query])

    # search
    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(metadata[idx])

    return results

In [38]:
results = semantic_search("What is RDF?")
results


[{'subject': 'data_semantics',
  'file_name': 'DS2526 - 2.2 - Knowledge Graphs and RDF - Part II.pdf',
  'chunk_id': 8,
  'chunk_text': 'rdf vocabularies for rdf 10 artificial intelligence  unimib 11 rdf: resource description framework artificial intelligence  unimib w3c recommendations  w3c: world wide web consortium  created to lead the web to its full potential by developing common protocols that promote its evolution and ensure its interoperability.  international industry consortium jointly run by the mit computer science and artificial intelligence laboratory (mit csail) in the usa, the european research consortium for informatics and mathematics (ercim) headquartered in france and keio university in japan. 350 organizations are members of w3c.  services provided by the consortium include: a repository of information',
  'chunk_length': 697},
 {'subject': 'data_semantics',
  'file_name': 'DS2526 - 2.2 - Knowledge Graphs and RDF - Part II.pdf',
  'chunk_id': 9,
  'chunk_text': 'ar

In [ ]:
print(type(chunks))
print(chunks[0].keys())

In [37]:
for i, result in enumerate(results):

    print("=" * 80)

    print(f"Result {i+1}")

    print("Subject:", result["subject"])

    print("File:", result["file_name"])

    print()

    print(result["chunk_text"][:1000])

Result 1
Subject: data_semantics
File: DS2526 - 2.2 - Knowledge Graphs and RDF - Part II.pdf

rdf vocabularies for rdf 10 artificial intelligence  unimib 11 rdf: resource description framework artificial intelligence  unimib w3c recommendations  w3c: world wide web consortium  created to lead the web to its full potential by developing common protocols that promote its evolution and ensure its interoperability.  international industry consortium jointly run by the mit computer science and artificial intelligence laboratory (mit csail) in the usa, the european research consortium for informatics and mathematics (ercim) headquartered in france and keio university in japan. 350 organizations are members of w3c.  services provided by the consortium include: a repository of information
Result 2
Subject: data_semantics
File: DS2526 - 2.2 - Knowledge Graphs and RDF - Part II.pdf

are members of w3c.  services provided by the consortium include: a repository of information about the world wide

In [26]:
semantic_search("What is clustering?")



[{'subject': 'machine_learning',
  'file_name': 'Machine Learning - [3] Clustering - 03 - Clustering Algorithms - part I - 0.pdf',
  'chunk_id': 0,
  'chunk_text': 'clustering: clustering algorithms  part i machine learning  fabio stella clustering clustering algorithms  part i fabio stella associate professor co department of informatics, systems and communication university of milano-bicocca clustering: clustering algorithms  part i machine learning  fabio stella the following concepts will be introduced: rationale of prototype-based clustering centroid k-means clustering  algorithm clustering algorithms clustering: clustering algorithms  part i machine learning  fabio stella 1 in prototype-based clustering, a cluster is a subset of objects (records) such that any object is closer to the prototype that defines the cluster to which it belongs to than',
  'chunk_length': 698},
 {'subject': 'machine_learning',
  'file_name': 'Machine Learning - [3] Clustering - 01 - Introduction - part 

In [27]:
semantic_search("Explain graph database")


[{'subject': 'data_semantics',
  'file_name': '00-survey.pdf',
  'chunk_id': 30,
  'chunk_text': 'sources from more trustworthy ones, and so forth. a graph dataset then consists of a set of named graphs and a default graph. each named graph is a pair of a graph id and a graph. the default graph is a graph without an id, and is referenced by default if a graph id is not specified. figure 2 provides an example where events and routes are stored in two named graphs, and the default graph manages meta-data about the named graphs (for a formal definition of a graph dataset, see definition b.3 in appendix b). we highlight that graph names can also be used as nodes in a graph. furthermore, nodes and edges can be repeated across graphs, where the same node in different graphs will typically',
  'chunk_length': 695},
 {'subject': 'data_management',
  'file_name': '5 Graphdb.pdf',
  'chunk_id': 0,
  'chunk_text': 'neo4j graph db 1 2 the graph model 1 nativenon native 2 design pattern 3 neo4j 4 q

In [29]:

semantic_search("What is ontology?")



[{'subject': 'data_semantics',
  'file_name': 'DS2526 - 2.4 - RDF Vocabularies.pdf',
  'chunk_id': 14,
  'chunk_text': '5 (2): 199220. krr  artificial intelligence  unimib 15 (axiomatic) ontologies an axiomatic ontology is a formal specification of a conceptualization by means of a language l, a set of logical axioms of l and a formal semantics that uniquely determines the meaning of the terms, i.e., the symbols of l. to determine the meaning of a term  to specify its unambiguous interpretation within a mathematical model axiomatic ontologies organize knowledge by defining:  symbols to represent individuals (e.g., entitites like elton john), classes (e.g., entity types like musical artist), relations (e.g., properties to describe entities like made)  the vocabulary  axioms to represent the logical',
  'chunk_length': 688},
 {'subject': 'data_semantics',
  'file_name': 'OM-State of the Art and Future Challenges.pdf',
  'chunk_id': 3,
  'chunk_text': 'matching. an ontology typically prov

In [30]:
semantic_search("What is overfitting?")

[{'subject': 'machine_learning',
  'file_name': 'Machine Learning - [2] Classification - 03 - Performance Evaluation - part I - 0 (1).pdf',
  'chunk_id': 13,
  'chunk_text': '(orange line) achieving the lowest value of the training error, achieves the highest value of the generalization error, while the classification model (blue line) achieving the highest value of the training error, achieves the lowest value of the generalization error. model overfitting consider a training set where two numeric input attributes are considered and a binary class attribute must be predicted. classification: performance evaluation  part i machine learning  fabio stella also the opposite behavior can emerge. 4 performance evaluation: underfitting 1 x 2 x classification: performance evaluation  part i machine learning  fabio stella 4 performance evaluation: underfitting 1 x 2 x 4',
  'chunk_length': 699},
 {'subject': 'machine_learning',
  'file_name': 'Machine Learning - [2] Classification - 03 - Perfo